<a href="https://colab.research.google.com/github/PunamBhatia/LogicMojo-AI-ML-April26-Punam/blob/main/NLP/EmotionsClassification_DistilBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#lets import required packages.
#We need HuggingFace datasets, transformers, evaluate and accelerate

import numpy as np
from datasets import load_dataset

%pip install evaluate
%pip install accelerate
%pip install transformers[torch]

import evaluate
from transformers import(
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


emotions_dataset= load_dataset("dair-ai/emotion")
print(emotions_dataset)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

...

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [2]:
print(emotions_dataset["train"][0])
label_names = emotions_dataset["train"].features["label"].names
print("Emotion Labels:", label_names)

{'text': 'i didnt feel humiliated', 'label': 0}
Emotion Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [3]:
#Tokenize
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
  return tokenizer(batch["text"], truncation=True, padding="max_length")

tokenized_emotions_dataset = emotions_dataset.map(tokenize_function, batched=True)
print(tokenized_emotions_dataset["train"][0])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': 'i didnt feel humiliated', 'label': 0, 'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [4]:
#load the pre-trained weights for the distilbert-base-uncased base model with 6 specific emotion categories
model= AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=6
)

# Load the standard accuracy metric using the evaluation library
accuracy_metric = evaluate.load("accuracy")

# Define a helper function to calculate accuracy during training loops
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
#lets define The Hugging Face Trainer
# Define optimization and output directories
training_args = TrainingArguments(
    output_dir="./emotion_model_results",  # Folder where model progress is saved
    eval_strategy="epoch",  # Check accuracy at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,  # Slow learning rate for fine-tuning
    per_device_train_batch_size=16,  # Process 16 samples at a time
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # 1 epoch is typically fast and sufficient for a tutorial
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Pass all variables to the automated Trainer pipeline
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_emotions_dataset["train"],
    eval_dataset=tokenized_emotions_dataset["validation"],
    compute_metrics=compute_metrics,
)

# Launch the training!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.270507,0.216475,0.921000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.5265767974853516, metrics={'train_runtime': 805.4593, 'train_samples_per_second': 19.864, 'train_steps_per_second': 1.242, 'total_flos': 2119629570048000.0, 'train_loss': 0.5265767974853516, 'epoch': 1.0})

In [6]:
#Lets test on sample
sample_sentence ="I can't believe I passed my final exam after studying all night!"


def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True )

    # Move inputs onto the active computing device (CPU or GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    logits = model(**inputs).logits

    # Identify the highest index value
    predicted_label_id = np.argmax(logits.detach().cpu().numpy(), axis=1)[0]

    print("Predicted Label ID:", predicted_label_id)

    # Map the numerical ID back to the corresponding word string
    predicted_emotion = label_names[predicted_label_id]
    return predicted_emotion

result = predict_emotion(sample_sentence)
print(f"Text: '{sample_sentence}' -> Predicted Emotion: {result}")

Predicted Label ID: 1
Text: 'I can't believe I passed my final exam after studying all night!' -> Predicted Emotion: joy
